# 01 · Prompt와 모델

**목표**: Gen AI Hub SDK로 LLM을 호출하고, message role과 system instruction이 출력에 미치는 영향을 비교합니다.

LLM은 다음 토큰을 예측하는 모델이며, 우리가 보내는 입력 전체를 prompt라고 합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Prompt Chatbot (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 system prompt와 모델을 바꾸며 대화를 이어가세요. 대화 history는 현재 브라우저 session에만 유지되며, AI Core에는 답변 생성 요청만 보냅니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>질문을 입력하고 답변을 확인합니다.</li>
<li>⚙️ 설정에서 system prompt 또는 모델을 바꿔 응답 형식을 비교합니다.</li>
<li>이 UI는 업무 상태를 변경하지 않습니다.</li>
</ol>
  </div>
</div>


In [1]:
from pathlib import Path
import json
from time import perf_counter
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lab_core import run_chat, run_chat_stream
from src.notebook_ui import build_chat_ui
chat_ui = build_chat_ui(run_chat, chat_stream_runner=run_chat_stream)
chat_ui.launch(inline=True, share=False, server_name='127.0.0.1', show_error=True, width='100%', height=720)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">메시지 role과 준비</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li><code>system</code>은 모델의 역할, 규칙, 언어, 출력 형식을 정합니다.</li>
<li><code>user</code>는 사용자의 질문 또는 요청입니다.</li>
<li><code>assistant</code>는 이전 모델 응답으로, 대화 history에 사용합니다.</li>
</ul>
<p style="margin: 7px 0;">아래 코드는 루트 <code>.env</code>의 <code>LLM_MODEL</code>을 사용합니다. UI를 실행한 뒤에는 선택 심화 자료로 코드 기반 비교를 확인하세요.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


In [2]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from lab_common.llm import chat_text
from lab_common.config import load_settings
from src.prompt_model.lab_core import compare_prompts, run_chat
settings = load_settings()
print(f'LLM model: {settings.llm_model}')

LLM model: gpt-5.6-luna


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 prompt 비교하기</h2>
<p style="margin: 7px 0;">이 아래 셀들은 UI와 같은 LLM 호출을 공개 코어 <code>compare_prompts()</code>로 확인하는 심화 자료입니다. 기본 실습에는 실행할 필요가 없습니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">기본 chat 응답</h3>
<p style="margin: 7px 0;">같은 질문을 기본 조건과 표 형식 system 조건으로 함께 호출하고, 먼저 기본 응답을 확인합니다.</p>
  </div>
</div>


In [4]:
question = 'LLM의 기본 아키텍처를 설명해줘.'
comparison_system_prompt = """너는 친절한 선생님이다.
초등학생도 이해할 수 있게 쉬운 한국어로 설명한다.
첫 줄에는 한 문장 요약을 쓴다.
그 아래에 핵심 내용을 bullet 3개로 정리한다."""

# comparison_system_prompt를 직접 바꿔 원하는 제약을 실험하세요.
comparison = compare_prompts(question, settings, system_prompt=comparison_system_prompt)
print(comparison['baseline'])

LLM(Large Language Model)의 기본 아키텍처는 대부분 **Transformer**를 기반으로 합니다. 특히 ChatGPT 같은 생성형 LLM은 주로 **Decoder-only Transformer** 구조를 사용합니다.

## 1. 전체 흐름

텍스트는 다음 과정을 거칩니다.

```text
문장
 → 토큰화
 → 토큰 임베딩 + 위치 정보
 → Transformer 블록 반복
 → 다음 토큰의 확률 계산
 → 토큰을 하나 선택해 문장 생성
```

예를 들어:

```text
"오늘 날씨가"
```

를 입력하면 모델은 다음 토큰으로 `"좋다"`, `"맑다"` 등의 확률을 계산하고, 그중 하나를 선택합니다. 이후 선택한 토큰을 다시 입력에 추가해 다음 토큰을 예측합니다.

---

## 2. 토큰화(Tokenization)

모델은 문장을 그대로 이해하지 못하므로 텍스트를 작은 단위인 **토큰(token)**으로 나눕니다.

```text
"나는 학교에 간다"
→ ["나는", "학교", "에", "간다"]
```

실제 모델에서는 단어 전체가 아니라 단어의 일부나 자주 등장하는 문자열 단위로 나뉠 수 있습니다.

각 토큰은 숫자 ID로 변환됩니다.

```text
["나는", "학교"] → [1024, 5831]
```

---

## 3. 임베딩(Embedding)

토큰 ID는 고차원 벡터로 변환됩니다.

```text
토큰 ID → 벡터
```

예를 들어 `"고양이"`라는 토큰은 수천 차원의 숫자 벡터로 표현됩니다. 이 벡터는 학습 과정에서 의미와 문법적 특성을 담도록 조정됩니다.

보통 다음 정보가 결합됩니다.

- **토큰 임베딩**: 어떤 단어 또는 토큰인지
- **위치 정보**: 문장에서 어디에 있는지

Transformer는 기본적으로 토큰의 순서를 직접 알지 못하기 때문에 위치 정보가 필요합니다.

---

## 4. Transformer 블록

LLM은 여러 개의 Transformer 블록을 층층이 쌓은 구조입니다. 각 

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">system prompt로 제약하기</h3>
<p style="margin: 7px 0;">같은 코어 호출에서 반환된 표 형식 결과를 확인합니다. 제목·표·bullet처럼 검증하기 쉬운 제약은 기본 응답과 차이를 더 분명하게 만듭니다.</p>
  </div>
</div>


In [5]:
print(comparison['constrained'])

LLM은 문장을 작은 단위로 바꾸고, 앞뒤 단어의 관계를 살펴 다음에 올 말을 예측하는 인공지능입니다.

- **입력 변환**: 문장을 토큰이라는 작은 단위로 나누고, 각 토큰을 숫자 벡터인 **임베딩**으로 바꿉니다.
- **문맥 이해**: **트랜스포머(Transformer)** 구조의 어텐션이 문장 속 단어들이 서로 어떤 관계인지 살핍니다. 예를 들어 “사과를 먹었다”에서 “사과”가 과일일 가능성을 문맥으로 판단합니다.
- **다음 단어 예측**: 여러 계산층을 거친 뒤, 지금까지의 내용을 바탕으로 다음 토큰의 확률을 계산하고, 이를 반복해 문장을 만들어 냅니다. LLM은 보통 이 과정을 매우 많은 데이터로 학습합니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 2</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">message 배열을 직접 구성하기</h2>
<p style="margin: 7px 0;">LLM API에는 하나의 문자열이 아니라 역할이 붙은 메시지 배열을 보냅니다. 아래에서 <code>system</code> 규칙, 이전 <code>user</code> 질문, 이전 <code>assistant</code> 답변, 새 <code>user</code> 질문의 순서를 확인하고 실행하세요.</p>
  </div>
</div>


In [6]:
messages = [
    {"role": "system", "content": "너는 친절한 교육 보조자다. 한국어로 간결하게 답한다."},
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
    {"role": "user", "content": "방금 답을 표로 정리해줘."},
]
for index, item in enumerate(messages, start=1):
    print(f'{index}. {item["role"]}: {item["content"]}')

reply = chat_text(settings, messages, settings.llm_model)
print()
print('응답:')
print(reply)

1. system: 너는 친절한 교육 보조자다. 한국어로 간결하게 답한다.
2. user: LLM의 장점을 두 가지 알려줘.
3. assistant: 자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다.
4. user: 방금 답을 표로 정리해줘.

응답:
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 적절히 답변할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 요약, 번역, 글쓰기, 정보 정리 등을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">대화 history가 맥락을 전달하는 방식</h2>
<p style="margin: 7px 0;">두 번째 질문만 보면 무엇을 표로 바꾸라는지 알 수 없습니다. 같은 질문을 history 없이, 그리고 history와 함께 보내 결과를 비교해 보세요.</p>
<p style="margin: 7px 0;"><strong>관찰:</strong> 코어는 최근 12개 메시지와 총 12,000자 안에서만 history를 보존합니다. 긴 대화에서는 중요한 맥락을 요약하거나 문서 검색으로 보완해야 합니다.</p>
  </div>
</div>


In [7]:
follow_up = '방금 답을 표로 정리해줘.'
history = [
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
]
without_history = run_chat(follow_up, settings)
with_history = run_chat(follow_up, settings, history=history)
print('[history 없음]')
print(without_history['reply'])
print()
print('[history 포함]')
print(with_history['reply'])

[history 없음]
표로 정리할 내용이 현재 대화에 보이지 않습니다. 방금 답변이나 정리할 내용을 보내주시면 표로 깔끔하게 정리해드릴게요.

[history 포함]
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 적절하게 답변할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 글쓰기, 요약, 번역, 교정 등 여러 텍스트 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">프롬프트로 JSON 형식 요청하고 검증하기</h2>
<p style="margin: 7px 0;">형식을 요청해도 항상 파싱 가능한 JSON이 반환된다고 보장할 수는 없습니다. 실행 결과가 파싱에 성공하는지 확인하고, 실패했다면 모델이 반환한 원문을 살펴보세요.</p>
  </div>
</div>


In [8]:
json_prompt = (
    '너는 교육 보조자다. 반드시 JSON 객체만 반환한다. Markdown 코드 블록이나 설명을 덧붙이지 않는다. '
    '키는 summary(문자열), keywords(문자열 배열)만 사용한다.'
)
json_outcome = run_chat(
    'LLM을 한 문장으로 요약하고 핵심 키워드 세 개를 추출해줘.',
    settings,
    system_prompt=json_prompt,
)
try:
    parsed = json.loads(json_outcome['reply'])
    print('JSON 파싱 성공:', parsed)
except json.JSONDecodeError:
    print('JSON 파싱 실패. 반환 원문을 확인하세요:')
    print(json_outcome['reply'])

JSON 파싱 성공: {'summary': 'LLM은 방대한 텍스트 데이터를 학습해 사람처럼 자연스러운 언어를 이해하고 생성하는 인공지능 모델이다.', 'keywords': ['대규모 언어 모델', '자연어 처리', '텍스트 생성']}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">동일 질문을 모델별로 비교하기</h2>
<p style="margin: 7px 0;">현재 모델과 AI Core에 배포된 다른 모델을 같은 질문으로 호출해 보세요. 응답 시간뿐 아니라 정확성, 지시 준수, 출력 형식을 함께 기록합니다. 배포되지 않은 모델명은 호출할 수 없습니다.</p>
  </div>
</div>


In [14]:
models_to_compare = [settings.llm_model]
# AI Core에 RUNNING 상태로 배포된 모델이 있다면 아래처럼 추가합니다.
models_to_compare.append('gpt-4.1')

comparison_question = 'SAP에서 2026년도에 출시한 제품은?'
for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = run_chat(comparison_question, settings, model=model_name)
    elapsed = perf_counter() - started_at
    print(f'[{model_name}] {elapsed:.1f}초')
    print(outcome['reply'])
    print('='*100 + '\n')

[gpt-5.6-luna] 7.0초
SAP는 매년 “신제품”을 한 번에 발표하기보다 기존 클라우드 제품의 정기 릴리스를 출시합니다. 따라서 **2026년에 새로 출시된 SAP 제품**과 **2026년 버전으로 업데이트된 제품**은 구분해야 합니다.

대표적인 2026년 릴리스는 다음과 같습니다.

- **SAP S/4HANA Cloud Public Edition 2602** — 2026년 상반기 릴리스
- **SAP S/4HANA Cloud Public Edition 2608** — 2026년 하반기 릴리스
- **SAP SuccessFactors H1/H2 2026 릴리스**
- **SAP Ariba, SAP Concur, SAP Fieldglass 등 SAP 클라우드 솔루션의 2026년 릴리스**
- **SAP Business Technology Platform(BTP) 및 SAP Business AI·Joule 관련 기능 업데이트**

다만 **SAP Business Data Cloud**처럼 이름이 새로 등장한 주요 제품은 **2025년에 발표된 제품**이며, 2026년 신제품은 기존 제품군의 확장·통합 형태가 많은 편입니다.

질문에서 말하는 “출시”가 **완전히 새로운 SAP 제품**인지, 아니면 **2026 버전으로 업데이트된 제품**인지에 따라 목록이 달라집니다.

[gpt-4.1] 4.4초
2024년 6월 기준으로, SAP가 2026년에 출시한 제품에 대한 공식 정보는 아직 공개되지 않았습니다. SAP는 정기적으로 새로운 버전의 SAP S/4HANA, SAP Business Technology Platform(BTP), SAP SuccessFactors 등 다양한 클라우드 및 온프레미스 솔루션을 출시하고 있지만, 2026년에 출시될 예정이거나 신규로 발표된 제품에 대한 자세한 정보는 현재 시점에서 확인할 수 없습니다.

제품 출시 및 로드맵 관련 최신 소식은 SAP 공식 웹사이트([SAP Newsroom](https://news.sap.com/

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">사내 FAQ 답변 도우미의 system prompt 설계</h2>
<p style="margin: 7px 0;">아래 셀에서 대상 독자, 말투, 답변 형식, 근거가 부족할 때의 행동을 포함한 system prompt를 작성하세요. 같은 질문을 기본 응답과 비교하고, 요구한 형식과 안전 규칙이 지켜졌는지 평가합니다.</p>
  </div>
</div>


In [21]:
faq_system_prompt = '''
TODO: 사내 FAQ 답변 도우미의 규칙을 작성하세요.
'''
faq_question = '출장비 정산 마감일이 언제인가요?'

# 작성 후 아래 주석을 해제해 기본 응답과 비교하세요.
# print(run_chat(faq_question, settings)['reply'])
# print(run_chat(faq_question, settings, system_prompt=faq_system_prompt)['reply'])